# 🧩 Class 2.1: `Z_LLM_ASK`, Designing the Interface
### GenAI & Agentic AI for SAP Professionals · Day 2 · Morning · 2 hours

**The idea in one line:** treat the AI like a new remote-enabled function module in your landscape, **`Z_LLM_ASK`**.

| A normal function module (SE37) | `Z_LLM_ASK` (the LLM) |
|---|---|
| Fixed, typed **IMPORTING** parameters | Accepts **any text** |
| Fixed **EXPORTING** structure | Returns **any text** |
| Exceptions defined in SE37 | No exceptions: it **always** answers, even when it shouldn't |

> 🔑 **An LLM is a function module with no interface. Prompt engineering = designing that interface.**

| Step | What we do | Cells |
|---|---|---|
| 1 | Call it with one vague parameter | ▶ 1 |
| 2 | Define the 5 interface parameters (**R-T-C-F-R**) | ▶ 2–5 |
| 3 | Regression test: remove one parameter | ▶ 6 |
| 4 | Calling patterns: zero-shot · few-shot · consumer view | ▶ 7–10 |
| 5 | Build your own interface | ▶ 11–14 |

**Runs locally on Ollama**, a local "RFC destination" at `localhost:11434`. The model is set in `bootcamp_config.json`. 🔒 Mock data only.

## ⚙️ ▶ CELL S: Setup
Like testing an RFC destination in SM59: checks that Ollama runs and the model exists.

In [ ]:
# ▶ CELL S · SETUP: test the local AI destination (Ollama). Run once.
import json, re, csv, time
from bootcamp_llm import *          # ask_ai, chat, ask_small, ask_json, show, parse_json, check_setup, last_usage, MODEL
from IPython.display import Markdown, display

DATA = json.load(open("../data/emails.json", encoding="utf-8"))
check_setup()
print("\n🤖", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10),
      f"· {last_usage['seconds']} s · model = {MODEL}")

---
# 1️⃣ A call with one vague parameter
### ▶ CELL 1: `CALL FUNCTION 'Z_LLM_ASK' EXPORTING text = 'Summarise this email.'`
✋ *You're the AP lead. Can you act on what comes back?*

In [ ]:
# ▶ CELL 1 · Call Z_LLM_ASK with one vague parameter
EMAIL_1 = DATA["EMAIL_1"]
print(EMAIL_1, "\n" + "─" * 70)

weak = ask_ai("Summarise this email.\n\n" + EMAIL_1, temperature=0)
show(weak)
print(f"\n⏱ {last_usage['seconds']} s · {len(weak.split())} words")

> 📄 You get text back, but **not what the caller needs**: no PO number, no escalation decision, no next step.
> It's like a BAPI returning one free-text message instead of a filled structure.

---
# 2️⃣ Define the interface: 5 parameters
| Prompt part | SAP object it behaves like | Our value |
|---|---|---|
| **R**ole | **PFCG role**: who is calling, and how they need it | SAP procurement analyst |
| **T**ask | **FM short text**: one function, one job | Summarise the dispute |
| **C**ontext | **SPRO customizing**: rules read at runtime | Above ₹5,00,000 → Finance Controller |
| **F**ormat | **SE11 structure**: fixed fields | Vendor · Invoice · PO · … |
| **R**ules | **Checks / exceptions**: `MESSAGE E…` | No guessing: "Not stated" |

> The model has **no SPRO and no SE11**. Everything it should know or follow, **you pass in the call**.

### ▶ CELL 2: Define each parameter separately

In [ ]:
# ▶ CELL 2 · Define the 5 interface parameters
ROLE    = "You are an SAP procurement analyst."
TASK    = "Task: Summarise this invoice dispute email."
CONTEXT = ("Context (company policy):\n"
           "- ₹5 lakh = ₹5,00,000.\n"
           "- If the disputed invoice amount is ABOVE ₹5,00,000 → escalate to the Finance Controller.\n"
           "- Otherwise → the AP team handles it.")
FORMAT  = ("Format:\n- Vendor:\n- Invoice number:\n- PO number:\n- Invoice amount:\n- Issue (1 line):\n"
           "- Escalate to Finance Controller? (Yes/No + reason):\n- Next action:")
RULES   = ('Rules:\n- If a value is not in the email, write "Not stated". Do not guess.\n'
           "- Keep each line under 20 words.")

def llm_call(*sections, email=EMAIL_1):
    """Assemble the interface parameters, then attach the email (the payload) between <<< >>> markers."""
    return "\n".join(sections) + "\n\nEmail:\n<<<\n" + email + "\n>>>"

print(llm_call(ROLE, TASK, CONTEXT, FORMAT, RULES))

### ▶ CELL 3: Version management, from v0.1 to v1.0
Same model, same email. Only the interface changes, like comparing versions in SE37.

In [ ]:
# ▶ CELL 3 · Version management: v0.1 → v1.0
versions = {
    "v0.1 · Role + Task":                    (ROLE, TASK),
    "v0.2 · + Structure (Format)":           (ROLE, TASK, FORMAT),
    "v1.0 · + Customizing + Checks":         (ROLE, TASK, CONTEXT, FORMAT, RULES),
}
for name, parts in versions.items():
    display(Markdown(f"### {name}"))
    show(ask_ai(llm_call(*parts), temperature=0))
    print(f"⏱ {last_usage['seconds']} s")

### ▶ CELL 4: Unit test: expected vs actual
Like an SE37 single test: does the result contain what the caller needs?

In [ ]:
# ▶ CELL 4 · Unit test: expected vs actual
EXPECTED = {"Invoice no.": "INV-77821", "PO no.": "4500012346", "Amount": "7,80,000",
              "Escalation": "Escalate", "Next action": "Next action"}

full = ask_ai(llm_call(ROLE, TASK, CONTEXT, FORMAT, RULES), temperature=0)

print(f"{'Check':14} | {'1 param':9} | {'5 params':9}")
print("─" * 40)
for label, needle in EXPECTED.items():
    w = "✅" if needle.lower() in weak.lower() else "❌"
    f = "✅" if needle.lower() in full.lower() else "❌"
    print(f"{label:14} | {w:^9} | {f:^9}")

### ▶ CELL 5: A vague rule vs an exact customizing value

In [ ]:
# ▶ CELL 5 · Vague text vs exact customizing value
VAGUE = "Context: Disputes on invoices above ₹5 lakh must go to the finance controller."
for name, ctx in [("😕 Vague", VAGUE), ("✅ Explicit", CONTEXT)]:
    answer = ask_ai(llm_call(ROLE, TASK, ctx, FORMAT, RULES), temperature=0)
    line = next((l for l in answer.splitlines() if "escalat" in l.lower()), "(no escalation line)")
    print(f"{name:11} → {line.strip()}")

> 🔎 **What we saw in testing (same model, same email):**
> - Vague rule, run A: *"Escalate? **No**: price dispute is below ₹5 lakh"* ❌ (₹7,80,000 is **above** ₹5 lakh)
> - Vague rule, run B: *"Escalate? **Yes**: price dispute"* ⚠️ (right answer, **wrong reason**)
> - Exact rule: *"Escalate? **Yes (Amount > ₹5,00,000)**"* ✅ on every run
>
> 🔑 Pass business rules the way they sit in a **customizing table**: exact value, exact threshold, exact owner. Not a sentence from a meeting.

---
# 3️⃣ Regression test
### ▶ CELL 6: Remove one parameter at a time, then re-run the test
*Like checking the impact of removing a parameter from a function module.*

In [ ]:
# ▶ CELL 6 · Regression test: remove one parameter
params = {"Role": ROLE, "Task": TASK, "Context": CONTEXT, "Format": FORMAT, "Rules": RULES}
for missing in params:
    kept = [v for k, v in params.items() if k != missing]
    answer = ask_ai(llm_call(*kept), temperature=0)
    score = sum(n.lower() in answer.lower() for n in EXPECTED.values())
    print(f"Without {missing:8} → {score}/{len(EXPECTED)} checks passed · {len(answer.split()):>3} words")

> 📉 Usually the **structure (Format)** is the parameter you can't drop: without it, fields disappear and the layout changes every run.

---
# 4️⃣ Calling patterns
| Pattern | SAP equivalent |
|---|---|
| **Zero-shot** | Calling the FM with no test data |
| **Few-shot** | Supplying sample entries, like the **test data directory** in SE37 |
| **Consumer view** | Same FM, different output per consumer, like **report variants** |

### ▶ CELL 7: Zero-shot, no examples

In [ ]:
# ▶ CELL 7 · Call without examples (zero-shot)
question = DATA["CLASSIFY_SET"]["C"]
show(ask_ai(f'Classify this email: "{question}"', temperature=0))

😕 Its own categories plus an essay: nothing a routing rule can use.

### ▶ CELL 8: Few-shot, 3 sample entries define the allowed values
*Like a domain with fixed values in SE11: the examples tell the model which values exist.*

In [ ]:
# ▶ CELL 8 · Call with 3 sample entries (few-shot)
FEW_SHOT = """Classify each email into exactly one category:
INVOICE_ISSUE, DELIVERY_DELAY, PRICE_QUERY, OTHER.
Reply with the category only.

Email: "The goods for PO 4500012345 haven't arrived yet." → DELIVERY_DELAY
Email: "Invoice 9001 shows a higher price than agreed." → INVOICE_ISSUE
Email: "What is your price for 500 units of bearings?" → PRICE_QUERY

Email: "{email}" →"""

print(ask_ai(FEW_SHOT.format(email=question), temperature=0, max_tokens=10))

### ▶ CELL 9: Unit test on 5 emails

In [ ]:
# ▶ CELL 9 · Unit test on 5 emails
score = 0
for k, email in DATA["CLASSIFY_SET"].items():
    label = ask_ai(FEW_SHOT.format(email=email), temperature=0, max_tokens=10).strip().strip("*.` ")
    ok = label == DATA["CLASSIFY_KEY"][k]
    score += ok
    print(f"{k}: {label:15} {'✅' if ok else '❌ expected ' + DATA['CLASSIFY_KEY'][k]}   {email[:55]}…")
print(f"\n🏁 Unit test: {score}/5 passed")

> 🚩 **Email D:** *"Please update our bank details"*. In SAP, bank data is a **sensitive field**: a change needs **dual control (FK08 / FK09 confirmation)**. An AI must never bypass that. (Day 4: guardrails.)

### ▶ CELL 10: Same function, three consumers

In [ ]:
# ▶ CELL 10 · Same function, three consumers
EMAIL_2 = DATA["EMAIL_2"]
consumers = {
    "🧑‍💼 Procurement analyst (internal)": "You are an SAP procurement analyst. Summarise for the internal team in 3 bullets.",
    "🤝 Vendor relations (external reply)": "You are a polite vendor relations manager. Draft a short reply to the vendor (max 5 lines).",
    "📊 CFO (one-line status)":            "You are writing for a CFO. Give a 1-line update.",
}
for name, instruction in consumers.items():
    display(Markdown(f"#### {name}"))
    show(ask_ai(instruction + "\n\n<<<\n" + EMAIL_2 + "\n>>>", temperature=0))

---
# 5️⃣ Build your own interface (40 min)
Change only the lines marked **← change me**, then press **Shift + Enter**.

### ▶ CELL 11: Task 1, define the interface for Email 3 (a quantity difference)

In [ ]:
# ▶ CELL 11 · Your interface for Email 3
EMAIL_3 = DATA["EMAIL_3"]
print(EMAIL_3, "\n" + "─" * 70)

my_role    = "You are an SAP accounts payable clerk."                               # ← change me
my_task    = "Task: Summarise this email and recommend an action."                  # ← change me
my_context = "Context: Quantity differences of more than 2% need a credit note."    # ← change me
my_format  = "Format:\n- Vendor:\n- Invoice:\n- Difference:\n- Credit note needed (Yes/No):\n- Next action:"   # ← change me
my_rules   = 'Rules: If a value is missing, write "Not stated". Max 15 words per line.'  # ← change me

show(ask_ai(llm_call(my_role, my_task, my_context, my_format, my_rules, email=EMAIL_3), temperature=0))

✅ *Check with Python:* 20 of 400 pieces = **5%** → above 2% → credit note **Yes**.
> 🧮 In our test run the model wrote *"approx 1.67%"*: **wrong calculation, right decision**. Never accept AI arithmetic unchecked (Class 2.2).

### ▶ CELL 12: Task 2, define your own categories (few-shot) for SAP support tickets

In [ ]:
# ▶ CELL 12 · Your own categories (few-shot)
MY_FEW_SHOT = """Classify the ticket as: MASTER_DATA, AUTHORISATION, PERFORMANCE, OTHER. Reply with the label only.
Ticket: "Please create vendor V500 in company code 1000." → MASTER_DATA
Ticket: "I get 'No authorisation for transaction ME21N'." → AUTHORISATION
Ticket: "{t}" →"""                                                              # ← change the labels/examples

my_tickets = ["Report ZMM_STOCK takes 20 minutes to run.",                      # ← change me
              "Need access to MIGO for plant 2000.",
              "Change the bank key for vendor V300."]
for t in my_tickets:
    print(f"{ask_ai(MY_FEW_SHOT.format(t=t), temperature=0, max_tokens=10).strip():15} ← {t}")

### ▶ CELL 13: Task 3, a different consumer of the same email

In [ ]:
# ▶ CELL 13 · Your consumer
my_persona = "You are a strict internal auditor. List 3 risks in this email, one line each."   # ← change me
show(ask_ai(my_persona + "\n\n<<<\n" + EMAIL_2 + "\n>>>", temperature=0))

### ▶ CELL 14: Task 4, Code Inspector for your prompt
*Like SCI/ATC: the AI scores your interface 0–2 per parameter.*

In [ ]:
# ▶ CELL 14 · Code Inspector for your prompt (score 0–2 per parameter)
prompt_to_review = "Summarise this email."                                       # ← paste your prompt here
show(ask_ai(f"""Score this prompt 0-2 on each of: Role, Task, Context, Format, Rules (max 10).
Show a small markdown table, then give ONE concrete improvement.
Prompt:
<<<
{prompt_to_review}
>>>""", temperature=0))

---
## ✅ Class 2.1 takeaways
1. An LLM is a **function module with no interface**. Your prompt defines it: Role · Task · Context · Format · Rules.
2. The model has **no customizing**. Pass rules with **exact values**.
3. **Few-shot** = sample entries that fix the allowed values. **Consumer view** = the right output for each caller.

> 🌉 **After lunch, Class 2.2:** `Z_LLM_ASK` goes to production. A **program** consumes its output, so we need **typed output, debugging and security**.

---
## 🛠️ Trainer troubleshooting (Ollama)
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model hasn't been pulled | In a terminal: `ollama pull <model name>` |
| `ModuleNotFoundError: bootcamp_llm` | The notebook was opened from a different folder | Open the **whole bundle folder** in VS Code or Antigravity, and keep the notebook inside `notebooks/` |
| Very slow first answer | The model is loading into memory | Normal. The next answers are faster. |
| Slow on every answer | The laptop is too small for the model | Choose a smaller `model` in `bootcamp_config.json` (e.g. `llama3.2:1b`), then **Restart kernel** |
| Answers differ from the slides | A different model or a new run | Expected: small local models vary. Discuss *why*. |
| Change the model mid-class | | Edit `bootcamp_config.json` → **Restart kernel** → run CELL S again |